In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin,urlparse,urldefrag
from datetime import datetime
from curl_cffi import requests as curl_requests
from curl_cffi.requests import exceptions as curl_exceptions
import random
import time

In [2]:

def create_session():
    session =  curl_requests.Session(
        impersonate="chrome120",  # Spoofs Chrome's exact TLS Cipher suite
        headers={
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.5",
            "Sec-Ch-Ua": '"Not_A Brand";v="8", "Chromium";v="120", "Google Chrome";v="120"',
            "Sec-Ch-Ua-Mobile": "?0",
            "Sec-Ch-Ua-Platform": '"Windows"',
            "Sec-Fetch-Dest": "document",
            "Sec-Fetch-Mode": "navigate",
            "Sec-Fetch-Site": "cross-site",
            "Upgrade-Insecure-Requests": "1",
        },
    )
    return session

def get_soup(url,session,referer="https://www.google.com/"):
    sleep_time=random.uniform(1, 4)  # Random sleep between 1 and 4 seconds
    time.sleep(sleep_time)
    
    session.headers.update({"Referer": referer})
    try:
        response = session.get(url,timeout=(3,10))
        response.raise_for_status()
    except curl_exceptions.Timeout:
        raise Exception(f"Request to {url} timed out.")
    except curl_exceptions.RequestException as e:
        raise Exception(f"Request to {url} failed: {e}")
    except Exception as e:
        raise Exception(f"An error occurred while fetching {url}: {e}")
    res_lower=response.text.lower()
    if "cf-turnstile" in res_lower or "just a moment" in res_lower:
        raise Exception("Encountered Cloudflare Turnstile challenge. Please solve it manually.")
    return BeautifulSoup(response.text, 'html.parser')

def extract_title(soup):
    title_tag = soup.find('title')
    if title_tag:
        return title_tag.get_text(strip=True)
    return None

def extract_links(soup, base_url):
    unique_links=set()
    seed_domain = urlparse(base_url).netloc
    root_word=base_url.replace("www.","")
    for link in soup.find_all('a'):
        href = link.get('href')
        if not href:
            continue
        full_url = urljoin(base_url, href)
        full_url, _ = urldefrag(full_url)
        
        if full_url.count(root_word) > 1:
            continue
        
        parsed = urlparse(full_url)
        if parsed.netloc == seed_domain:
            unique_links.add(full_url)
            
    return list(unique_links)
    

def extract_content(soup):
    container=soup.find("article")
    if container:
        paragraphs = container.find_all('p')
        return "\n".join(p.get_text(strip=True) for p in paragraphs)
    paragraphs = soup.find_all('p')
    return "\n".join(p.get_text(strip=True) for p in paragraphs)

def create_document(url,active_session,current_referer):
    soup = get_soup(url, active_session, referer=current_referer)
    if soup is None:
        raise Exception(f"Failed to retrieve or parse the content from {url}.")
    
    title = extract_title(soup)
    links = extract_links(soup, url)
    content = extract_content(soup)

    document = {
        'url': url,
        'title': title,
        'links': links,
        'content': content,
        'metadata': {
            'created_at': datetime.now().isoformat(),
        }
    }

    return document

def crawler(url,session,last_referer):
    visited=set()
    queued={url }
    to_visit=[url]
    documents=[]
    
    while to_visit:
        current_url=to_visit.pop(0)
        if current_url in visited:
            continue
        print(f"crawling : {current_url}")
        try:
            document=create_document(current_url,session,last_referer)
            if not document:
                visited.add(current_url)
                continue
            documents.append(document)
            visited.add(current_url)
            for link in document['links']:
                if link not in visited and link not in queued:
                    queued.add(link)
                    to_visit.append(link)
            last_referer=current_url
        except Exception as e:
            print(f"Error processing {current_url}: {e}")
    
    return documents
    

In [3]:
master_urls=[ "https://www.learnpytorch.io/",]#"https://d2l.ai/",
live_brower = create_session()

last_referer = "https://www.google.com/"

# scraped_documents = []
# for url in master_urls:
#     try:
#         document = create_document(url, live_brower, last_referer)
#         scraped_documents.append(document)
#         last_referer = url  # Update the referer for the next request
#     except Exception as e:
#         print(f"Error scraping {url}: {e}")


In [4]:
crawled_documents = crawler(master_urls[0], live_brower, last_referer)


crawling : https://www.learnpytorch.io/
crawling : https://www.learnpytorch.io/03_pytorch_computer_vision/
crawling : https://www.learnpytorch.io/pytorch_2_intro/
crawling : https://www.learnpytorch.io/pytorch_most_common_errors/
crawling : https://www.learnpytorch.io/04_pytorch_custom_datasets/
Error processing https://www.learnpytorch.io/04_pytorch_custom_datasets/: Request to https://www.learnpytorch.io/04_pytorch_custom_datasets/ timed out.
crawling : https://www.learnpytorch.io/pytorch_extra_resources/
crawling : https://www.learnpytorch.io/01_pytorch_workflow/
crawling : https://www.learnpytorch.io/05_pytorch_going_modular/
crawling : https://www.learnpytorch.io/02_pytorch_classification/
Error processing https://www.learnpytorch.io/02_pytorch_classification/: Request to https://www.learnpytorch.io/02_pytorch_classification/ timed out.
crawling : https://www.learnpytorch.io/09_pytorch_model_deployment/
crawling : https://www.learnpytorch.io/07_pytorch_experiment_tracking/
Error p

In [5]:
crawled_documents

[{'url': 'https://www.learnpytorch.io/',
  'title': 'Zero to Mastery Learn PyTorch for Deep Learning',
  'links': ['https://www.learnpytorch.io/03_pytorch_computer_vision/',
   'https://www.learnpytorch.io/pytorch_2_intro/',
   'https://www.learnpytorch.io/pytorch_most_common_errors/',
   'https://www.learnpytorch.io/04_pytorch_custom_datasets/',
   'https://www.learnpytorch.io/pytorch_extra_resources/',
   'https://www.learnpytorch.io/01_pytorch_workflow/',
   'https://www.learnpytorch.io/05_pytorch_going_modular/',
   'https://www.learnpytorch.io/',
   'https://www.learnpytorch.io/02_pytorch_classification/',
   'https://www.learnpytorch.io/09_pytorch_model_deployment/',
   'https://www.learnpytorch.io/07_pytorch_experiment_tracking/',
   'https://www.learnpytorch.io/00_pytorch_fundamentals/',
   'https://www.learnpytorch.io/08_pytorch_paper_replicating/',
   'https://www.learnpytorch.io/06_pytorch_transfer_learning/',
   'https://www.learnpytorch.io/pytorch_cheatsheet/'],
  'content

In [6]:
# d2l = create_document()
# print(d2l)

In [7]:
# d2l_links = d2l['links']
# print(f"Extracted {len(d2l_links)} links:")
# for link in d2l_links:
#     print(f"- link: {link}")

In [8]:
# content_pages = [
#     url for url in d2l_links
#     if "chapter_" in url
# ]

# print(len(content_pages))

In [9]:
# learn_pytorch=create_document()
# learn_pytorch_links=learn_pytorch['links']
# print(f"Extracted {len(learn_pytorch_links)} links:")
# for link in learn_pytorch_links:
#     print(f"- link: {link}")

In [10]:
# pytorch=create_document("https://docs.pytorch.org/docs/2.12/index.html")
# pytorch_links=pytorch['links']
# print(f"Extracted {len(pytorch_links)} links:")
# for link in pytorch_links:
#     print(f"- link: {link}")